# Transaction Fraud Detection — Final Model

**Goal**: maximize Total Utility = 50·TP − 100·FN − 5·FP.

**Approach**
1. Sort all transactions by time and build features that use **only past information** (no label leakage):
   cardholder history, trailing 1-day / 7-day relative frequency of category / merchant / hour / state / job
   across all transactions, and trailing 7-day amount level per category.
2. Normal transactions of a given category appear in bulk only in certain months, so identity / time proxy
   features (lat/long, job, age, month, ...) are dropped. "Relative frequency" features express whether
   a value is unusually rare in the current window.
3. The threshold is chosen with **month-blocked cross-validation** (6 folds x 3 months), not a single
   "train on the past, validate on the future" split.
4. CatBoost is refit on all labelled data, and the best OOF threshold converts probabilities to 0/1
   to produce `submission.csv`.

Local OOF Utility ~ 492,410 (best threshold ~ 0.0175).

In [1]:
from pathlib import Path
import time

import numpy as np
import pandas as pd

# ---- Configuration: edit paths as needed ----
DATA_DIR = Path("data/transaction-fraud-detection")   # contains fraudTrain.csv / fraudTest.csv
OUT_DIR = Path("outputs")
RUN_CV = True            # False skips cross-validation (~15 min) and uses FALLBACK_THRESHOLD
FALLBACK_THRESHOLD = 0.0175
SEED = 42
CB_PARAMS = dict(iterations=800, learning_rate=0.05, depth=8)

OUT_DIR.mkdir(exist_ok=True)

## 1. Utility function and threshold search

In [2]:
def utility(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    tp = ((y_true == 1) & (y_pred == 1)).sum()
    fn = ((y_true == 1) & (y_pred == 0)).sum()
    fp = ((y_true == 0) & (y_pred == 1)).sum()
    return int(50 * tp - 100 * fn - 5 * fp)


def best_threshold(y, p):
    """Scan every cut-off of the ranked probabilities; return (best threshold, max Utility)."""
    y = np.asarray(y)
    order = np.argsort(-p, kind="stable")
    ys, ps = y[order], p[order]
    tp = np.cumsum(ys)
    fp = np.cumsum(1 - ys)
    pos = y.sum()
    u = np.concatenate([[-100 * pos], 50 * tp - 100 * (pos - tp) - 5 * fp])
    k = int(np.argmax(u))
    if k == 0:
        return 1.0 + 1e-9, int(u[0])
    nxt = ps[k] if k < len(ps) else 0.0
    return float((ps[k - 1] + nxt) / 2), int(u[k])

## 2. Load data
- `cc_num` is corrupted by scientific notation in the test file, so cardholders are identified by
  `first|last|dob|street`.
- `unix_time` is used as the common timestamp.

In [3]:
def load():
    tr = pd.read_csv(DATA_DIR / "fraudTrain.csv", dtype={"cc_num": str, "zip": str})
    te = pd.read_csv(DATA_DIR / "fraudTest.csv", dtype={"cc_num": str, "zip": str})
    tr["dob"] = pd.to_datetime(tr["dob"], format="%Y-%m-%d")
    te["dob"] = pd.to_datetime(te["dob"], format="%m/%d/%Y")
    tr["is_test"], te["is_test"] = 0, 1
    te["is_fraud"] = -1
    tr["row_order"] = np.arange(len(tr))
    te["row_order"] = np.arange(len(te)) + len(tr)
    df = pd.concat([tr, te], ignore_index=True)
    df["card"] = (df["first"] + "|" + df["last"] + "|" + df["dob"].dt.strftime("%Y%m%d")
                  + "|" + df["street"])
    df["ts"] = pd.to_datetime(df["unix_time"], unit="s")
    return df.sort_values(["unix_time", "row_order"], kind="stable").reset_index(drop=True)


raw = load()
print(raw.shape, raw["is_test"].value_counts().to_dict())

(1872839, 27) {0: 1402996, 1: 469843}


## 3. Feature engineering (causal: only earlier transactions, no labels)

In [4]:
def haversine(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, (lat1, lon1, lat2, lon2))
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 6371 * 2 * np.arcsin(np.sqrt(a))


REL_DIMS = ["category", "merchant", "hour", "state", "job"]


def build_features(df):
    f = df.copy()
    f["log_amt"] = np.log1p(f["amt"])
    f["hour"] = f["ts"].dt.hour
    f["dow"] = f["ts"].dt.dayofweek
    f["is_weekend"] = (f["dow"] >= 5).astype(int)
    f["dist_home_merch"] = haversine(f["lat"], f["long"], f["merch_lat"], f["merch_long"])

    n = len(f)
    t = f["unix_time"].to_numpy(dtype=np.int64)
    amt = f["amt"].to_numpy(dtype=float)
    mlat, mlon = f["merch_lat"].to_numpy(), f["merch_long"].to_numpy()
    names = ["secs_since_prev", "cnt_1h", "cnt_24h", "sum_24h", "mean_amt_prior",
             "amt_ratio_prior", "dist_prev_merch", "speed_kmh"]
    cols = {k: np.full(n, np.nan) for k in names}
    # Cardholder history: each transaction only sees earlier ones of the same cardholder
    for idx in f.groupby("card", sort=False).indices.values():
        idx = np.sort(idx)
        tt, aa = t[idx], amt[idx]
        k = len(idx)
        pos = np.arange(k)
        cs = np.concatenate([[0.0], np.cumsum(aa)])
        for w, key in ((3600, "cnt_1h"), (86400, "cnt_24h")):
            left = np.searchsorted(tt, tt - w, side="left")
            cols[key][idx] = pos - left
            if w == 86400:
                cols["sum_24h"][idx] = cs[pos] - cs[left]
        mean = np.where(pos > 0, cs[pos] / np.maximum(pos, 1), np.nan)
        cols["mean_amt_prior"][idx] = mean
        cols["amt_ratio_prior"][idx] = aa / (mean + 1e-6)
        if k > 1:
            dt = np.diff(tt).astype(float)
            d = haversine(mlat[idx[:-1]], mlon[idx[:-1]], mlat[idx[1:]], mlon[idx[1:]])
            cols["secs_since_prev"][idx[1:]] = dt
            cols["dist_prev_merch"][idx[1:]] = d
            cols["speed_kmh"][idx[1:]] = d / np.maximum(dt, 60) * 3600
    for k_, v in cols.items():
        f[k_] = v
    f["new_merchant"] = (f.groupby(["card", "merchant"]).cumcount() == 0).astype(int)
    f["new_category"] = (f.groupby(["card", "category"]).cumcount() == 0).astype(int)

    # Trailing 1-day / 7-day relative frequency of the same value among ALL transactions (no labels)
    g_all = np.arange(n)
    for dim in REL_DIMS:
        for w, tag in ((86400, "1d"), (7 * 86400, "7d")):
            tot = g_all - np.searchsorted(t, t - w, side="left")
            cnt = np.zeros(n)
            for idx in f.groupby(dim, sort=False).indices.values():
                idx = np.sort(idx)
                tt = t[idx]
                cnt[idx] = np.arange(len(idx)) - np.searchsorted(tt, tt - w, side="left")
            f[f"rel_{dim}_{tag}"] = (cnt + 0.5) / (tot + 1)
            f[f"lcnt_{dim}_{tag}"] = np.log1p(cnt)
    return f


def extra_features(f):
    """More cardholder-history features + trailing 7-day amount level per category."""
    t = f["unix_time"].to_numpy(dtype=np.int64)
    amt = f["amt"].to_numpy(dtype=float)
    n = len(f)
    out = {k: np.full(n, np.nan) for k in
           ["card_max_prior", "amt_over_max", "nmerch_24h", "ncat_24h", "secs_same_cat",
            "amt_rank_card", "std_amt_prior", "amt_z", "night_frac_prior", "mean_amt_7d",
            "amt_ratio_7d"]}
    night = f["hour"].isin([22, 23, 0, 1, 2, 3]).to_numpy().astype(float)
    mer = f["merchant"].astype("category").cat.codes.to_numpy()
    cat = f["category"].astype("category").cat.codes.to_numpy()
    for idx in f.groupby("card", sort=False).indices.values():
        idx = np.sort(idx)
        tt, aa, nn = t[idx], amt[idx], night[idx]
        k = len(idx)
        pos = np.arange(k)
        prev_max = np.concatenate([[np.nan], np.maximum.accumulate(aa)[:-1]])
        out["card_max_prior"][idx] = prev_max
        out["amt_over_max"][idx] = aa / (prev_max + 1e-6)
        cs = np.concatenate([[0.0], np.cumsum(aa)])
        cs2 = np.concatenate([[0.0], np.cumsum(aa ** 2)])
        with np.errstate(invalid="ignore", divide="ignore"):
            mean = cs[pos] / pos
            var = cs2[pos] / pos - mean ** 2
        sd = np.sqrt(np.maximum(var, 0))
        out["std_amt_prior"][idx] = np.where(pos > 1, sd, np.nan)
        out["amt_z"][idx] = np.where(pos > 1, (aa - mean) / (sd + 1.0), np.nan)
        cn = np.concatenate([[0.0], np.cumsum(nn)])
        out["night_frac_prior"][idx] = np.where(pos > 0, cn[pos] / np.maximum(pos, 1), np.nan)
        l7 = np.searchsorted(tt, tt - 7 * 86400, side="left")
        m7 = np.where(pos - l7 > 0, (cs[pos] - cs[l7]) / np.maximum(pos - l7, 1), np.nan)
        out["mean_amt_7d"][idx] = m7
        out["amt_ratio_7d"][idx] = aa / (m7 + 1e-6)
        l24 = np.searchsorted(tt, tt - 86400, side="left")
        mm, cc = mer[idx], cat[idx]
        last_cat = {}
        for i in range(k):
            out["nmerch_24h"][idx[i]] = len(set(mm[l24[i]:i]))
            out["ncat_24h"][idx[i]] = len(set(cc[l24[i]:i]))
            lc = last_cat.get(cc[i])
            out["secs_same_cat"][idx[i]] = tt[i] - lc if lc is not None else np.nan
            last_cat[cc[i]] = tt[i]
        out["amt_rank_card"][idx] = [(aa[:i] < aa[i]).mean() if i else np.nan for i in range(k)]
    for k_, v in out.items():
        f[k_] = v

    ca = np.full(n, np.nan)
    cr = np.full(n, np.nan)
    for idx in f.groupby("category", sort=False).indices.values():
        idx = np.sort(idx)
        tt, aa = t[idx], amt[idx]
        cs = np.concatenate([[0.0], np.cumsum(np.log1p(aa))])
        l = np.searchsorted(tt, tt - 7 * 86400, side="left")
        pos = np.arange(len(idx))
        mean = np.where(pos - l > 0, (cs[pos] - cs[l]) / np.maximum(pos - l, 1), np.nan)
        ca[idx] = mean
        cr[idx] = np.log1p(aa[:]) - mean
    f["cat_lamt_7d"] = ca
    f["amt_vs_cat_7d"] = cr
    return f


NUM_BASE = ["amt", "log_amt", "hour", "dow", "is_weekend", "dist_home_merch"]
NUM_HIST = ["secs_since_prev", "cnt_1h", "cnt_24h", "sum_24h", "mean_amt_prior",
            "amt_ratio_prior", "dist_prev_merch", "speed_kmh", "new_merchant", "new_category"]
NUM_REL = [f"{p}_{d}_{w}" for d in REL_DIMS for w in ("1d", "7d") for p in ("rel", "lcnt")]
EXTRA = ["card_max_prior", "amt_over_max", "nmerch_24h", "ncat_24h", "secs_same_cat",
         "amt_rank_card", "std_amt_prior", "amt_z", "night_frac_prior", "mean_amt_7d",
         "amt_ratio_7d", "cat_lamt_7d", "amt_vs_cat_7d"]
CAT_FEATS = ["category"]
FEATS = NUM_BASE + NUM_HIST + CAT_FEATS + NUM_REL + EXTRA
print(len(FEATS), "features")

50 features


In [5]:
t0 = time.time()
df = extra_features(build_features(raw))
print(f"features built in {time.time() - t0:.0f}s")
lab = df[df["is_test"] == 0].reset_index(drop=True)
te = df[df["is_test"] == 1].reset_index(drop=True)
y = lab["is_fraud"].astype(int).to_numpy()
print("labelled:", len(lab), "fraud rate:", round(y.mean(), 5), "| test:", len(te))

features built in 18s
labelled: 1402996 fraud rate: 0.00755 | test: 469843


## 4. Month-blocked cross-validation and threshold selection
The 18 labelled months are split into 6 folds of 3 consecutive months. Each fold is predicted by a model
trained on the other folds, giving out-of-fold (OOF) probabilities for every labelled row; the threshold
that maximizes Utility on the OOF predictions is then selected.

In [6]:
from catboost import CatBoostClassifier, Pool


def make_model():
    return CatBoostClassifier(random_seed=SEED, verbose=0, thread_count=-1,
                              allow_writing_files=False, **CB_PARAMS)


if RUN_CV:
    month_idx = (lab["ts"].dt.year - 2024) * 12 + lab["ts"].dt.month - 1
    fold = (month_idx // 3).to_numpy()
    oof = np.zeros(len(lab))
    for k in range(6):
        va = fold == k
        m = make_model()
        m.fit(Pool(lab.loc[~va, FEATS], y[~va], cat_features=CAT_FEATS))
        oof[va] = m.predict_proba(Pool(lab.loc[va, FEATS], cat_features=CAT_FEATS))[:, 1]
        print(f"fold {k} done", flush=True)
    THRESHOLD, u = best_threshold(y, oof)
    print(f"\npooled OOF: best threshold={THRESHOLD:.4f}, Utility={u} (theoretical reference 0.0323)")
    for k in range(6):
        va = fold == k
        print(f"  fold {k}: Utility={utility(y[va], (oof[va] >= THRESHOLD).astype(int))}")
    for thr in (0.01, 0.0175, 0.025, 0.04, 0.06):
        print(f"  fixed thr {thr}: Utility={utility(y, (oof >= thr).astype(int))}")
    pd.DataFrame({"ID": lab["ID"], "fold": fold, "y": y, "oof": oof}).to_csv(
        OUT_DIR / "oof.csv", index=False)
else:
    THRESHOLD = FALLBACK_THRESHOLD
    print("skip CV, threshold =", THRESHOLD)

fold 0 done
fold 1 done
fold 2 done
fold 3 done
fold 4 done
fold 5 done

pooled OOF: best threshold=0.0175, Utility=492410 (theoretical reference 0.0323)
  fold 0: Utility=64675
  fold 1: Utility=80860
  fold 2: Utility=88155
  fold 3: Utility=93015
  fold 4: Utility=88910
  fold 5: Utility=76795
  fixed thr 0.01: Utility=489210
  fixed thr 0.0175: Utility=492270
  fixed thr 0.025: Utility=491780
  fixed thr 0.04: Utility=490275
  fixed thr 0.06: Utility=486180


## 5. Refit on all labelled data and write submission.csv

In [7]:
final = make_model()
final.fit(Pool(lab[FEATS], y, cat_features=CAT_FEATS))

prob = pd.Series(final.predict_proba(Pool(te[FEATS], cat_features=CAT_FEATS))[:, 1],
                 index=te["ID"].to_numpy())
ids = pd.read_csv(DATA_DIR / "fraudTest.csv", usecols=["ID"])["ID"]   # keep the original test-file order
prob = prob.loc[ids]
submission = pd.DataFrame({"ID": ids.to_numpy(), "is_fraud": (prob.to_numpy() >= THRESHOLD).astype(int)})

assert len(submission) == len(ids) and submission["ID"].is_unique
assert set(submission["is_fraud"].unique()) <= {0, 1}
submission.to_csv(OUT_DIR / "submission.csv", index=False)
prob.rename("prob").to_csv(OUT_DIR / "test_probabilities.csv", index_label="ID")
print(f"rows={len(submission)}, flagged={submission['is_fraud'].mean():.4%}, "
      f"expected frauds (sum of probs)~{prob.sum():.0f}")

rows=469843, flagged=2.9823%, expected frauds (sum of probs)~10612


## 6. Feature importance

In [8]:
imp = pd.Series(final.get_feature_importance(), index=FEATS).sort_values(ascending=False)
imp.head(20)

hour                11.042384
amt                  9.976212
log_amt              8.373092
mean_amt_7d          4.578887
sum_24h              4.054413
mean_amt_prior       3.953699
rel_hour_7d          3.914319
nmerch_24h           2.995343
card_max_prior       2.735336
ncat_24h             2.711864
lcnt_hour_7d         2.704621
rel_category_1d      2.704204
cnt_24h              2.573513
cat_lamt_7d          2.317794
night_frac_prior     2.137174
amt_vs_cat_7d        2.107982
category             2.096524
secs_same_cat        2.064880
amt_ratio_prior      1.942617
lcnt_category_7d     1.880915
dtype: float64